In [ ]:
import pandas as pd
import numpy as np
import glob
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from math import radians, sin, cos, acos

pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)


In [ ]:
def calc_distance(lat1,lon1,lat2,lon2):
    if lat1-lat2==0:
        dist = 0
    else:
        lat1 = radians(lat1)
        lon1 = radians(lon1)
        lat2 = radians(lat2)
        lon2 = radians(lon2)
        dist = 6371.01 * acos(sin(lat1)*sin(lat2) + cos(lat1)*cos(lat2)*cos(lon1 - lon2))
    return dist

In [ ]:
fname = 'icesat2_ooi-rs03axps-sf03a-3c-parada301_2024-03-21T22_10_31Z.pkl'
gdf = pd.read_pickle('data/'+fname)
gdf

In [ ]:
# load time, lat and lon
fnames = glob.glob('ice*.pkl')
fnames.sort()
for jj in range(len(fnames)):
    gdf = pd.read_pickle(fnames[jj])
    # sort by photon collection time
    gdf = gdf.sort_index()
    
    # pull out lat, lon, and calculate distance from first photon
    lat = gdf.geometry.y.to_numpy()
    lon = gdf.geometry.x.to_numpy()
    gdf['latitude']= lat
    gdf['longitude']= lon
    gdf['dist_along_km'] = [calc_distance(lat[0],lon[0],lat[jj],lon[jj]) for jj in range(len(lat))]
    
    # remove unnecessary data
    gdf = gdf.drop(columns=['region', 'sc_orient','rgt','cycle','background_rate','landcover','relief','snowcover','atl08_class','spot','geometry'])
    
    # save as mat for use with Emily's Kd calculation code
    gdf.to_csv(fnames[jj][:-4]+'_formatlab.csv')

In [ ]:
# #confirm that above is doing what I expect...

# fig = plt.figure()
# plt.scatter(along_dist_km,height)

# fig = plt.figure()
# plt.scatter(longitude,latitude,c=height)

# fig = plt.figure()
# gdf.plot(column='height',aspect=1/10)